# Imputation Environment Check

Run the next cell to verify that required libraries are available for:
- Mean imputation (`SimpleImputer`)
- kNN imputation (`KNNImputer`)
- MICE and SoftImpute workflows (via `hyperimpute`)
- Optional research repos (`GRAPE`, `DiffPuter`)

In [1]:
import sys
import pathlib
import importlib
from importlib import metadata

# ---- self-contained DiffPuter path setup ----------------------------------
# This makes the cell work on its own. If you also have a separate path-setup
# cell, that's fine — running it twice is a no-op.
HERE = pathlib.Path.cwd()
for cand in [
    HERE / "DiffPuter",
    HERE / "external" / "DiffPuter",
    HERE.parent / "DiffPuter",
]:
    if (cand / "main.py").is_file():
        for p in [cand, cand / "baselines", cand / "baselines" / "GRAPE"]:
            sp = str(p.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
        break

# Module name (for import) -> distribution name (for version lookup).
# These differ for some packages (sklearn vs scikit-learn, yaml vs PyYAML,
# ot vs POT).
required_modules = {
    # core scientific
    "numpy":         "numpy",
    "pandas":        "pandas",
    "scipy":         "scipy",
    "sklearn":       "scikit-learn",
    "matplotlib":    "matplotlib",
    "statsmodels":   "statsmodels",

    # baselines + SOTA
    "fancyimpute":   "fancyimpute",     # Mean (SimpleFill), kNN, SoftImpute
    "hyperimpute":   "hyperimpute",     # HyperImpute + MICE plugin

    # deep learning + GRAPE deps
    "torch":             "torch",
    "torch_geometric":   "torch_geometric",
    "h5py":              "h5py",
    "networkx":          "networkx",

    # DiffPuter deps
    "ot":            "POT",             # POT distributes as POT, imports as ot
    "FrEIA":         "FrEIA",
    "timm":          "timm",
    "yaml":          "PyYAML",

    # notebook
    "ipykernel":     "ipykernel",
}

# Modules that are nice-to-have but the env still works without them.
optional_modules = {
    "torch_scatter": "torch_scatter",   # GRAPE only
}

# Note: GRAPE and DiffPuter are loaded as local source clones, not pip packages.


def check_module(module_name: str, package_name: str):
    try:
        importlib.import_module(module_name)
    except Exception as e:
        return "MISSING", str(e)
    try:
        version = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        version = "unknown"
    return "OK", version


print("Required packages")
print("-" * 60)
missing = []
for module_name, package_name in required_modules.items():
    status, info = check_module(module_name, package_name)
    marker = "✓" if status == "OK" else "✗"
    print(f"  {marker} {package_name:18} {status:8} {info}")
    if status == "MISSING":
        missing.append(package_name)

print("\nOptional packages")
print("-" * 60)
for module_name, package_name in optional_modules.items():
    status, info = check_module(module_name, package_name)
    marker = "✓" if status == "OK" else "○"
    print(f"  {marker} {package_name:18} {status:8} {info}")

print("\nSmoke-test imports for imputation APIs")
print("-" * 60)

api_checks = [
    ("Mean / kNN (sklearn)",
     lambda: __import__("sklearn.impute", fromlist=["SimpleImputer", "KNNImputer"])),
    ("MICE (sklearn IterativeImputer)",
     lambda: (
         __import__("sklearn.experimental", fromlist=["enable_iterative_imputer"]),
         __import__("sklearn.impute",       fromlist=["IterativeImputer"]),
     )),
    ("SoftImpute / KNN / SimpleFill (fancyimpute)",
     lambda: __import__("fancyimpute", fromlist=["SoftImpute", "KNN", "SimpleFill"])),
    ("HyperImpute (hyperimpute.plugins.imputers.Imputers)",
     lambda: __import__("hyperimpute.plugins.imputers", fromlist=["Imputers"])),
    ("PyTorch Geometric (for GRAPE)",
     lambda: __import__("torch_geometric")),
]

for label, fn in api_checks:
    try:
        fn()
        print(f"  ✓ {label}: OK")
    except Exception as e:
        print(f"  ✗ {label}: FAILED | {e}")

print("\nLocal repo modules (DiffPuter / GRAPE)")
print("-" * 60)
repo_checks = [
    ("DiffPuter dataset",   "dataset",          ["load_dataset", "get_eval", "mean_std"]),
    ("DiffPuter diffusion", "diffusion_utils",  ["sample_step", "impute_mask", "EDMLoss"]),
    ("DiffPuter model",     "model",            ["MLPDiffusion", "Model"]),
    ("GRAPE training",      "training.gnn_mdi", ["train_gnn_mdi"]),
]
for label, mod, names in repo_checks:
    try:
        m = importlib.import_module(mod)
        for n in names:
            getattr(m, n)
        print(f"  ✓ {label}: OK ({mod})")
    except Exception as e:
        print(f"  ✗ {label}: FAILED | {type(e).__name__}: {e}")

print()
if missing:
    print(f"⚠  Missing required packages: {', '.join(missing)}")
    print("   Re-run: pip install -r requirements-base.txt")
else:
    print("All required packages present.")

Required packages
------------------------------------------------------------
  ✓ numpy              OK       1.26.4
  ✓ pandas             OK       2.3.3
  ✓ scipy              OK       1.17.1
  ✓ scikit-learn       OK       1.8.0
  ✓ matplotlib         OK       3.10.9
  ✓ statsmodels        OK       0.14.6
  ✓ fancyimpute        OK       0.7.0
  ✓ hyperimpute        OK       0.1.17
  ✓ torch              OK       2.11.0
  ✓ torch_geometric    OK       2.7.0
  ✓ h5py               OK       3.14.0
  ✓ networkx           OK       3.6.1


I0000 00:00:1777986327.082300    1177 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777986345.544536    1177 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


  ✓ POT                OK       0.9.6.post1
  ✓ FrEIA              OK       0.2
  ✓ timm               OK       1.0.26
  ✓ PyYAML             OK       6.0.3
  ✓ ipykernel          OK       7.2.0

Optional packages
------------------------------------------------------------
  ✓ torch_scatter      OK       2.1.2+pt211cu130

Smoke-test imports for imputation APIs
------------------------------------------------------------
  ✓ Mean / kNN (sklearn): OK
  ✓ MICE (sklearn IterativeImputer): OK
  ✓ SoftImpute / KNN / SimpleFill (fancyimpute): OK
  ✓ HyperImpute (hyperimpute.plugins.imputers.Imputers): OK
  ✓ PyTorch Geometric (for GRAPE): OK

Local repo modules (DiffPuter / GRAPE)
------------------------------------------------------------
  ✓ DiffPuter dataset: OK (dataset)
  ✓ DiffPuter diffusion: OK (diffusion_utils)
  ✓ DiffPuter model: OK (model)
  ✓ GRAPE training: OK (training.gnn_mdi)

All required packages present.


# Dataset Analysis
Perform simple data exploration: Shape, Columns, Data types, Missing values, First few rows

In [5]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Load the dataset
df = pd.read_csv('Scenario5/scenario5.csv')

# Drop unnamed columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Convert relative paths to absolute for reference
base_dir = Path('Scenario5')

# Data exploration
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst few rows:")
df.head()

Shape: (2300, 15)

Columns: ['index', 'unit1_rgb', 'unit1_pwr_60ghz', 'unit1_loc', 'unit2_loc', 'unit1_beam_index', 'seq_index', 'time_stamp[UTC]', 'unit2_direction', 'unit2_num_sat', 'unit2_sat_used', 'unit2_fix_type', 'unit2_DGPS', 'unit2_PDOP', 'unit2_HDOP']

Data types:
index                 int64
unit1_rgb            object
unit1_pwr_60ghz      object
unit1_loc            object
unit2_loc            object
unit1_beam_index      int64
seq_index             int64
time_stamp[UTC]      object
unit2_direction       int64
unit2_num_sat         int64
unit2_sat_used       object
unit2_fix_type       object
unit2_DGPS           object
unit2_PDOP          float64
unit2_HDOP          float64
dtype: object

Missing values:
index               0
unit1_rgb           0
unit1_pwr_60ghz     0
unit1_loc           0
unit2_loc           0
unit1_beam_index    0
seq_index           0
time_stamp[UTC]     0
unit2_direction     0
unit2_num_sat       0
unit2_sat_used      0
unit2_fix_type      0
unit2_DGPS

,index,unit1_rgb,unit1_pwr_60ghz,unit1_loc,unit2_loc,unit1_beam_index,seq_index,time_stamp[UTC],unit2_direction,unit2_num_sat,unit2_sat_used,unit2_fix_type,unit2_DGPS,unit2_PDOP,unit2_HDOP
0,1,./unit1/camera_data/image_BS1_259_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_0.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_0.txt,60,1,['03-20-31-142'],1,21,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
1,2,./unit1/camera_data/image_BS1_260_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_1.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_1.txt,60,1,['03-20-31-284'],1,24,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
2,3,./unit1/camera_data/image_BS1_261_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_2.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_2.txt,58,1,['03-20-31-426'],1,14,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
3,4,./unit1/camera_data/image_BS1_262_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_3.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_3.txt,59,1,['03-20-31-568'],1,21,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
4,5,./unit1/camera_data/image_BS1_263_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_4.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_4.txt,60,1,['03-20-31-710'],1,21,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
